# HPFL Simulation Notebook

In [ ]:
!pip install torch torchvision numpy scikit-learn

## 1. Imports

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset, DataLoader
from collections import OrderedDict
import time
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import copy

## 2. Configuration (`config.py`)

In [ ]:
config = {
    # Data and model configuration
    "dataset": "MNIST",  # "MNIST" or "FEMNIST"
    "scenario": "strong",  # "weak", "medium", or "strong" Non-IID

    # Federated learning parameters
    "rounds": 10, # Reduced for quick testing in Colab
    "local_epochs": 2, # Reduced for quick testing
    "learning_rate": 0.01,

    # Hierarchical structure
    "num_clients": 50, # Reduced for quick testing
    "num_uavs": 5,
    "max_clients_per_uav": 10,

    # Clustering parameters
    "initial_clusters_k": 3,
    "cluster_threshold": 0.5,  # Similarity threshold for merging/splitting

    # DCS (Dynamic Client Selection) parameters
    "dcs_weights": {
        "alpha": 0.25,  # Communication quality
        "beta": 0.25,   # Compute capacity
        "gamma": 0.25,  # Data significance
        "delta": 0.25,  # Contribution (loss improvement)
    },

    # Experiment mode
    "baseline_mode": "HPFL",  # "FedAvg", "DCS_only", "Clustering_only", "HPFL"
}

## 3. Model Definitions (`models.py`)

In [ ]:
class SimpleCNN(nn.Module):
    """A simple CNN to act as the shared backbone.
    
    Consists of two convolutional layers and one fully connected layer.
    """
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=5)
        self.fc1 = nn.Linear(1024, 512) # 1024 = 64 * 4 * 4

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 1024) # Flatten the tensor
        x = F.relu(self.fc1(x))
        return x

class PersonalizedHead(nn.Module):
    """A personalized classifier head for each client.
    
    A simple fully-connected layer that takes the output of the backbone.
    """
    def __init__(self, input_dim=512, output_dim=10):
        super(PersonalizedHead, self).__init__()
        self.fc2 = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.fc2(x)

class PersonalizedModel(nn.Module):
    """Combines the shared backbone and a personalized head.
    """
    def __init__(self, backbone, head):
        super(PersonalizedModel, self).__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, x):
        features = self.backbone(x)
        output = self.head(features)
        return output

def get_model_parameters(model):
    """Returns a model's state_dict.
    """
    return model.state_dict()

def set_model_parameters(model, params):
    """Sets a model's parameters from a state_dict.
    """
    model.load_state_dict(params)

## 4. Data Loading and Partitioning (`data.py`)

In [ ]:
def load_dataset(root, dataset_name, train=True):
    """Loads the specified dataset.

    Args:
        root (str): The root directory where the dataset is stored.
        dataset_name (str): The name of the dataset to load (e.g., "MNIST").
        train (bool): Whether to load the training or test set.

    Returns:
        A PyTorch Dataset.
    """
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    if dataset_name == "MNIST":
        dataset = torchvision.datasets.MNIST(
            root=root,
            train=train,
            download=True,
            transform=transform
        )
    elif dataset_name == "FEMNIST":
        print("FEMNIST is not available in torchvision, using MNIST instead.")
        dataset = torchvision.datasets.MNIST(
            root=root,
            train=train,
            download=True,
            transform=transform
        )        
    else:
        raise ValueError(f"Dataset {dataset_name} not supported.")

    return dataset

def partition_data(dataset, num_clients, scenario="strong"):
    """Partitions the dataset for a number of clients to simulate Non-IID data.

    Args:
        dataset: The dataset to partition.
        num_clients (int): The number of clients.
        scenario (str): The Non-IID scenario ("strong", "medium", "weak").

    Returns:
        A list of DataLoaders, one for each client.
    """
    if scenario == "strong":
        # Strong Non-IID: Each client gets data from only a few classes.
        # Simple implementation: Sort data by label and distribute chunks.
        labels = dataset.targets.numpy()
        sorted_indices = np.argsort(labels)
        
        # Shuffle within sorted chunks to add some randomness
        # This is a simple way to create shards.
        shards = np.array_split(sorted_indices, num_clients * 2) # Create more shards than clients
        np.random.shuffle(shards)
        
        client_data_indices = [np.concatenate(shards[i*2:(i+1)*2]) for i in range(num_clients)]

    else:
        # For now, only strong Non-IID is implemented
        raise NotImplementedError(f"Scenario '{scenario}' is not implemented yet.")

    client_dataloaders = []
    for indices in client_data_indices:
        client_images = dataset.data[indices]
        client_labels = dataset.targets[indices]
        
        # Add a channel dimension for grayscale images
        if len(client_images.shape) == 3:
            client_images = client_images.unsqueeze(1)
            
        # Normalize manually as transform is not applied on subset
        client_images = client_images.float() / 255.0
        client_images = (client_images - 0.5) / 0.5

        tensor_dataset = TensorDataset(client_images, client_labels)
        dataloader = DataLoader(tensor_dataset, batch_size=32, shuffle=True)
        client_dataloaders.append(dataloader)

    return client_dataloaders

## 5. Clustering Utilities (`clustering.py`)

In [ ]:
def model_to_vector(model_state):
    """Flattens a model's state_dict into a single vector.

    Args:
        model_state (OrderedDict): The model's state_dict.

    Returns:
        A 1D numpy array representing the model's parameters.
    """
    # Concatenate all parameters into a single tensor
    params = [param.cpu().numpy().flatten() for param in model_state.values()]
    return np.concatenate(params)

def compute_similarity_matrix(vectors):
    """Computes the pairwise cosine similarity between a list of vectors.

    Args:
        vectors (list or np.ndarray): A list of 1D numpy arrays.

    Returns:
        A 2D numpy array representing the similarity matrix.
    """
    return cosine_similarity(vectors)

def cluster_assignment(sim_matrix, max_k):
    """Groups models into clusters based on their similarity matrix.

    This implementation uses K-means clustering on the similarity vectors.

    Args:
        sim_matrix (np.ndarray): The similarity matrix.
        max_k (int): The maximum number of clusters to form.

    Returns:
        A numpy array of cluster labels for each model.
    """
    if sim_matrix.shape[0] <= max_k:
        # If there are fewer models than max_k, assign each to its own cluster
        return np.arange(sim_matrix.shape[0])

    kmeans = KMeans(n_clusters=max_k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(sim_matrix)
    
    return cluster_labels

## 6. DCS Utilities (`dcs.py`)

In [ ]:
def normalize_features(values):
    """Scales a list of values to the range [0, 1].

    Args:
        values (list or np.ndarray): The values to normalize.

    Returns:
        A numpy array of normalized values.
    """
    values = np.array(values)
    min_val = values.min()
    max_val = values.max()
    
    if max_val - min_val > 0:
        return (values - min_val) / (max_val - min_val)
    else:
        return np.zeros(values.shape) # All values are the same

def compute_scores(clients, weights):
    """Computes selection scores for a list of clients.

    Args:
        clients (list): A list of Client objects.
        weights (dict): A dictionary of weights for scoring (alpha, beta, gamma, delta).

    Returns:
        A list of tuples, where each tuple contains a client and their score.
    """
    if not clients:
        return []

    # Extract features for normalization
    comm_qualities = [client.comm_quality for client in clients]
    compute_powers = [client.compute_power for client in clients]
    data_significances = [client.data_significance for client in clients]
    contributions = [client.last_loss if client.last_loss is not None else 0 for client in clients]

    # Normalize features
    norm_q = normalize_features(comm_qualities)
    norm_c = normalize_features(compute_powers)
    norm_d = normalize_features(data_significances)
    norm_g = normalize_features(contributions)

    scores = []
    for i, client in enumerate(clients):
        score = (
            weights['alpha'] * norm_q[i] +
            weights['beta'] * norm_c[i] +
            weights['gamma'] * norm_d[i] +
            weights['delta'] * norm_g[i]
        )
        scores.append((client, score))
    
    return scores

## 7. Client Representation (`client.py`)

In [ ]:
class Client:
    """Represents a single client in the federated learning system."""
    def __init__(self, client_id, dataloader, backbone, head):
        self.client_id = client_id
        self.dataloader = dataloader
        self.model = PersonalizedModel(backbone, head)
        
        # Simulate client heterogeneity
        self.compute_power = np.random.uniform(0.5, 1.5) # Relative speed
        self.comm_quality = np.random.uniform(0.5, 1.5)  # Relative quality
        self.data_significance = len(self.dataloader.dataset)
        self.last_loss = None

    def local_train(self, shared_state, epochs, lr):
        """Performs local training on the client's data.

        Args:
            shared_state (dict): The state_dict of the shared backbone model.
            epochs (int): The number of local epochs to train for.
            lr (float): The learning rate for the optimizer.

        Returns:
            A tuple containing:
            - The updated shared model parameter deltas.
            - The updated personal head parameters.
            - The time taken for training.
            - The number of data samples used.
        """
        set_model_parameters(self.model.backbone, shared_state)
        self.model.train()
        
        optimizer = optim.SGD(self.model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()
        
        start_time = time.time()
        
        initial_loss = 0
        for epoch in range(epochs):
            epoch_loss = 0.0
            for images, labels in self.dataloader:
                optimizer.zero_grad()
                outputs = self.model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            if epoch == 0:
                initial_loss = epoch_loss / len(self.dataloader)

        end_time = time.time()
        training_time = (end_time - start_time) / self.compute_power
        
        final_loss = epoch_loss / len(self.dataloader)
        self.last_loss = initial_loss - final_loss # Contribution score

        # Calculate parameter deltas for the backbone
        updated_backbone_params = get_model_parameters(self.model.backbone)
        deltas = {key: updated_backbone_params[key] - shared_state[key] for key in shared_state}
        
        return deltas, get_model_parameters(self.model.head), training_time, len(self.dataloader.dataset)

    def compute_score(self, weights):
        """Computes the selection score for this client.

        The score is a weighted sum of communication quality, compute capacity,
        data significance, and contribution (loss improvement).
        
        S = alpha * q + beta * c + gamma * d + delta * g

        Args:
            weights (dict): A dictionary with weights for each component.

        Returns:
            The client's selection score.
        """
        # These values should be normalized across all clients by the UAV
        q = self.comm_quality
        c = self.compute_power
        d = self.data_significance
        g = self.last_loss if self.last_loss is not None else 0
        
        score = (
            weights['alpha'] * q +
            weights['beta'] * c +
            weights['gamma'] * d +
            weights['delta'] * g
        )
        
        return score

## 8. UAV Representation (`uav.py`)

In [ ]:
class UAV:
    """Represents a UAV that aggregates updates from a group of clients."""
    def __init__(self, uav_id, clients):
        self.uav_id = uav_id
        self.clients = clients
        self.shared_model_state = None # This will be updated by the satellite

    def select_clients(self, m, weights):
        """Selects the top m clients based on their DCS scores.

        Args:
            m (int): The number of clients to select.
            weights (dict): The weights for the DCS scoring formula.

        Returns:
            A list of the selected Client objects.
        """
        if not self.clients:
            return []
            
        client_scores = compute_scores(self.clients, weights)
        
        # Sort clients by score in descending order
        sorted_clients = sorted(client_scores, key=lambda x: x[1], reverse=True)
        
        # Select the top m clients
        selected_clients = [client for client, score in sorted_clients[:m]]
        
        return selected_clients

    def aggregate_updates(self, client_updates):
        """Aggregates parameter updates from selected clients.

        This implementation uses Federated Averaging (FedAvg), where updates are
        weighted by the number of data samples each client used for training.

        Args:
            client_updates (list): A list of tuples, where each tuple contains
                                   (deltas, num_samples).

        Returns:
            An OrderedDict with the aggregated model parameter deltas.
        """
        if not client_updates:
            return OrderedDict()

        total_samples = sum(num_samples for _, num_samples in client_updates)
        if total_samples == 0:
            return OrderedDict()

        # Initialize aggregated deltas with zeros
        aggregated_deltas = OrderedDict()
        for key in client_updates[0][0].keys():
            aggregated_deltas[key] = 0.0

        # Perform weighted averaging
        for deltas, num_samples in client_updates:
            weight = num_samples / total_samples
            for key in deltas:
                aggregated_deltas[key] += deltas[key] * weight
        
        return aggregated_deltas

## 9. Satellite Representation (`satellite.py`)

In [ ]:
class Satellite:
    """Represents the satellite responsible for global clustering and aggregation."""
    def __init__(self, initial_clusters_k):
        self.initial_clusters_k = initial_clusters_k
        self.uav_models = [] # List of model state_dicts from UAVs

    def cluster_models(self, uav_models):
        """Clusters UAV models based on parameter similarity.

        Args:
            uav_models (list): A list of model state_dicts from the UAVs.

        Returns:
            A dictionary mapping cluster labels to lists of UAV models.
        """
        self.uav_models = uav_models
        if not self.uav_models:
            return {}

        # Convert models to vectors
        model_vectors = [model_to_vector(model) for model in self.uav_models]
        
        # Compute similarity matrix
        sim_matrix = compute_similarity_matrix(model_vectors)
        
        # Get cluster assignments
        cluster_labels = cluster_assignment(sim_matrix, self.initial_clusters_k)
        
        # Group models by cluster
        clusters = {}
        for i, label in enumerate(cluster_labels):
            if label not in clusters:
                clusters[label] = []
            clusters[label].append(self.uav_models[i])
            
        return clusters

    def aggregate_clusters(self, clusters):
        """Aggregates models within each cluster to create global cluster models.

        Args:
            clusters (dict): A dictionary mapping cluster labels to lists of models.

        Returns:
            A dictionary mapping cluster labels to the aggregated global model state_dict.
        """
        aggregated_cluster_models = {}
        for label, models in clusters.items():
            if not models:
                continue

            # Initialize with the first model's structure
            aggregated_model = OrderedDict()
            for key in models[0].keys():
                aggregated_model[key] = 0.0

            # Simple averaging of model parameters
            num_models = len(models)
            for model in models:
                for key in model:
                    aggregated_model[key] += model[key] / num_models
            
            aggregated_cluster_models[label] = aggregated_model
            
        return aggregated_cluster_models

    def broadcast_global_model(self, cluster_models):
        """Distributes cluster-specific global models back to UAVs.
        
        (This is a conceptual step; in the simulation, the trainer will handle this.)
        """
        pass

## 10. Metrics and Logging (`metrics.py`)

In [ ]:
def compute_personalized_accuracy(client, test_loader):
    """Computes the accuracy of a client's personalized model on a test set.

    Args:
        client (Client): The client whose model should be evaluated.
        test_loader (DataLoader): The DataLoader for the test set.

    Returns:
        The accuracy of the model on the test set.
    """
    client.model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = client.model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    return correct / total

def compute_global_accuracy(global_model, global_test_loader):
    """Computes the accuracy of a global model on a global test set.

    Args:
        global_model (nn.Module): The global model to evaluate.
        global_test_loader (DataLoader): The DataLoader for the global test set.

    Returns:
        The accuracy of the model on the test set.
    """
    global_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in global_test_loader:
            outputs = global_model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    return correct / total

def estimate_comm_cost(payload_size_bytes, num_clients, is_uplink=True):
    """A simple function to estimate communication cost.
    
    (This is a placeholder for a more complex model.)
    """
    # Simple assumption: cost is proportional to payload size and number of clients
    base_cost_per_byte = 1e-6 # Arbitrary cost unit
    return payload_size_bytes * num_clients * base_cost_per_byte

def log_round_metrics(round_idx, metrics):
    """Logs the metrics for a given round to the console.
    
    Args:
        round_idx (int): The current round number.
        metrics (dict): A dictionary of metrics to log.
    """
    log_message = f"Round {round_idx+1}:"
    for key, value in metrics.items():
        log_message += f" | {key}: {value:.4f}"
    print(log_message)

## 11. Training Loop (`trainer.py`)

In [ ]:
def initialize_simulation(config):
    """Sets up the simulation environment.

    Args:
        config (dict): The experiment configuration.

    Returns:
        A tuple containing the list of clients, UAVs, the satellite, and the global test loader.
    """
    print("Initializing simulation...")
    # Load data
    train_dataset = load_dataset("./data", config["dataset"], train=True)
    test_dataset = load_dataset("./data", config["dataset"], train=False)
    client_dataloaders = partition_data(train_dataset, config["num_clients"], config["scenario"])
    global_test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128)

    # Create models
    shared_backbone = SimpleCNN()
    initial_shared_state = get_model_parameters(shared_backbone)

    # Create clients
    clients = []
    for i in range(config["num_clients"]):
        personal_head = PersonalizedHead()
        # We need to create a deepcopy of the backbone for each client to avoid sharing layers
        client = Client(f"client_{i}", client_dataloaders[i], copy.deepcopy(shared_backbone), personal_head)
        clients.append(client)

    # Create UAVs and assign clients
    uavs = []
    clients_per_uav = config["num_clients"] // config["num_uavs"]
    for i in range(config["num_uavs"]):
        start_idx = i * clients_per_uav
        end_idx = (i + 1) * clients_per_uav
        uav_clients = clients[start_idx:end_idx]
        uav = UAV(f"uav_{i}", uav_clients)
        uav.shared_model_state = copy.deepcopy(initial_shared_state)
        uavs.append(uav)

    # Create satellite
    satellite = Satellite(config["initial_clusters_k"])
    
    print("Initialization complete.")
    return clients, uavs, satellite, global_test_loader

def run_round(round_idx, uavs, satellite, config):
    """Runs a single round of federated learning.

    Args:
        round_idx (int): The current round index.
        uavs (list): The list of UAVs.
        satellite (Satellite): The satellite object.
        config (dict): The experiment configuration.
    """
    print(f"--- Round {round_idx+1} ---")
    uav_aggregated_models = []
    total_comm_cost = 0
    total_time_cost = 0

    for uav in uavs:
        # 1. Client Selection (DCS)
        selected_clients = uav.select_clients(m=config["max_clients_per_uav"], weights=config["dcs_weights"])
        
        # 2. Local Training
        client_updates = []
        round_training_times = []
        for client in selected_clients:
            deltas, _, training_time, num_samples = client.local_train(
                uav.shared_model_state, config["local_epochs"], config["learning_rate"]
            )
            client_updates.append((deltas, num_samples))
            round_training_times.append(training_time)
        
        if not client_updates:
            continue

        # 3. UAV Aggregation
        aggregated_deltas = uav.aggregate_updates(client_updates)
        
        # Apply aggregated deltas to the UAV's model
        current_uav_model = copy.deepcopy(uav.shared_model_state)
        for key in aggregated_deltas:
            current_uav_model[key] += aggregated_deltas[key]
        uav_aggregated_models.append(current_uav_model)
        
        # Update time cost (max training time in the round)
        total_time_cost += max(round_training_times) if round_training_times else 0

    # 4. Satellite Clustering and Aggregation
    if config["baseline_mode"] in ["Clustering_only", "HPFL"]:
        clusters = satellite.cluster_models(uav_aggregated_models)
        cluster_models = satellite.aggregate_clusters(clusters)
    else: # FedAvg or DCS_only (simple aggregation)
        clusters = {"global": uav_aggregated_models}
        cluster_models = satellite.aggregate_clusters(clusters)

    # 5. Model Update
    if not cluster_models:
        print("No models to update in this round.")
        return
        
    # In a real scenario, UAVs would be assigned to a cluster.
    # Here, we simplify and give all UAVs the first cluster's model (or the global one).
    global_model_state = list(cluster_models.values())[0]
    for uav in uavs:
        uav.shared_model_state = copy.deepcopy(global_model_state)

    # Update clients with the new shared state for the next round
    for uav in uavs:
        for client in uav.clients:
            set_model_parameters(client.model.backbone, uav.shared_model_state)


def run_experiment(config):
    """Runs the full HPFL simulation experiment."""
    clients, uavs, satellite, global_test_loader = initialize_simulation(config)
    
    for r in range(config["rounds"]):
        run_round(r, uavs, satellite, config)
        
        # Evaluate metrics
        all_accuracies = []
        for client in clients:
            # Use a subset of test data for quick personalized eval
            # In a real scenario, each client has its own test set.
            acc = compute_personalized_accuracy(client, global_test_loader)
            all_accuracies.append(acc)
        
        if not all_accuracies:
            avg_personalized_acc = 0
        else:
            avg_personalized_acc = sum(all_accuracies) / len(all_accuracies)
        
        metrics = {"Personalized Accuracy": avg_personalized_acc}
        log_round_metrics(r, metrics)

## 12. Run the Simulation

In [ ]:
run_experiment(config)